# shuttle-c0-vs-c4: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('28_shuttle-c0-vs-c4_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 16


,classifier,method,seed,fold,error
40,svm,adasyn,42,3,Not any neigbours belong to the majority class...
49,knn,adasyn,42,3,Not any neigbours belong to the majority class...
58,svm,adasyn,42,4,Not any neigbours belong to the majority class...
67,knn,adasyn,42,4,Not any neigbours belong to the majority class...
76,svm,adasyn,42,5,Not any neigbours belong to the majority class...
85,knn,adasyn,42,5,Not any neigbours belong to the majority class...
130,svm,adasyn,137,3,Not any neigbours belong to the majority class...
139,knn,adasyn,137,3,Not any neigbours belong to the majority class...
184,rf,adasyn,42,3,RuntimeError: Not any neigbours belong to the ...
193,lr,adasyn,42,3,RuntimeError: Not any neigbours belong to the ...


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,98.695,97.972,98.519,97.533,99.883,98.708,99.967,99.999,98.333
smote,99.508,98.414,97.719,99.200,99.824,99.512,99.967,99.999,98.961
bsmote,98.695,97.972,98.519,97.533,99.883,98.708,99.967,99.999,98.333
ros,99.537,98.792,98.431,99.200,99.883,99.541,99.967,99.999,99.164
adasyn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
casb,99.739,98.996,98.431,99.600,99.883,99.741,100.000,100.000,99.367
casb_relaxed,99.710,98.619,97.719,99.600,99.824,99.712,99.983,99.999,99.164
casb_edited,99.710,98.619,97.719,99.600,99.824,99.712,99.983,99.999,99.164
casb_ros,99.537,98.792,98.431,99.200,99.883,99.541,100.000,100.000,99.164


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,99.596,99.592,100.0,99.2,100.0,99.6,100.0,100.0,99.594
smote,99.596,99.592,100.0,99.2,100.0,99.6,100.0,100.0,99.594
bsmote,99.596,99.592,100.0,99.2,100.0,99.6,100.0,100.0,99.594
ros,99.596,99.592,100.0,99.2,100.0,99.6,100.0,100.0,99.594
adasyn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
casb,99.596,99.592,100.0,99.2,100.0,99.6,100.0,100.0,99.594
casb_relaxed,99.596,99.592,100.0,99.2,100.0,99.6,100.0,100.0,99.594
casb_edited,99.596,99.592,100.0,99.2,100.0,99.6,100.0,100.0,99.594
casb_ros,99.596,99.592,100.0,99.2,100.0,99.6,100.0,100.0,99.594


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
smote,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
bsmote,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
ros,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
adasyn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
casb,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
casb_relaxed,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
casb_edited,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
casb_ros,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,99.596,99.592,100.0,99.2,100.0,99.6,99.255,99.200,99.594
smote,99.596,99.592,100.0,99.2,100.0,99.6,99.255,99.200,99.594
bsmote,99.596,99.592,100.0,99.2,100.0,99.6,99.255,99.200,99.594
ros,99.596,99.592,100.0,99.2,100.0,99.6,99.255,99.200,99.594
adasyn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
casb,99.596,99.592,100.0,99.2,100.0,99.6,99.255,99.200,99.594
casb_relaxed,99.596,99.592,100.0,99.2,100.0,99.6,99.255,99.200,99.594
casb_edited,99.596,99.592,100.0,99.2,100.0,99.6,99.255,99.200,99.594
casb_ros,99.596,99.592,100.0,99.2,100.0,99.6,99.257,99.238,99.594


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,99.472,99.289,99.630,98.983,99.971,99.477,99.805,99.800,99.380
smote,99.675,99.400,99.430,99.400,99.956,99.678,99.805,99.800,99.537
bsmote,99.472,99.289,99.630,98.983,99.971,99.477,99.805,99.800,99.380
ros,99.682,99.494,99.608,99.400,99.971,99.685,99.805,99.800,99.588
adasyn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
casb,99.733,99.545,99.608,99.500,99.971,99.735,99.814,99.800,99.639
casb_relaxed,99.725,99.451,99.430,99.500,99.956,99.728,99.809,99.800,99.588
casb_edited,99.725,99.451,99.430,99.500,99.956,99.728,99.809,99.800,99.588
casb_ros,99.682,99.494,99.608,99.400,99.971,99.685,99.814,99.809,99.588


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,result
0,gmean,Incomplete coverage
1,f1,Incomplete coverage
2,precision,Incomplete coverage
3,recall,Incomplete coverage


## knn: per-metric family comparison

,metric,result
0,gmean,Incomplete coverage
1,f1,Incomplete coverage
2,precision,Incomplete coverage
3,recall,Incomplete coverage


## rf: per-metric family comparison

,metric,result
0,gmean,Incomplete coverage
1,f1,Incomplete coverage
2,precision,Incomplete coverage
3,recall,Incomplete coverage


... (3 outputs left out)

,metric,result
0,gmean,Incomplete coverage
1,f1,Incomplete coverage
2,precision,Incomplete coverage
3,recall,Incomplete coverage


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.978945,0.978723,1.000000,0.958333,1.000000,0.979167,0.998333,0.999939,0.978834,NaN
1,42,1,svm,smote,0.50,0.998537,0.979592,0.960000,1.000000,0.997076,0.998538,0.998333,0.999939,0.989064,NaN
2,42,1,svm,bsmote,0.50,0.978945,0.978723,1.000000,0.958333,1.000000,0.979167,0.998333,0.999939,0.978834,NaN
3,42,1,svm,ros,0.50,0.998537,0.979592,0.960000,1.000000,0.997076,0.998538,0.998333,0.999939,0.989064,NaN
4,42,1,svm,adasyn,0.50,0.998537,0.979592,0.960000,1.000000,0.997076,0.998538,1.000000,1.000000,0.989064,NaN
5,42,1,svm,casb,0.50,0.998537,0.979592,0.960000,1.000000,0.997076,0.998538,1.000000,1.000000,0.989064,NaN
6,42,1,svm,casb_relaxed,0.75,0.998537,0.979592,0.960000,1.000000,0.997076,0.998538,1.000000,1.000000,0.989064,NaN
7,42,1,svm,casb_edited,0.75,0.998537,0.979592,0.960000,1.000000,0.997076,0.998538,1.000000,1.000000,0.989064,NaN
8,42,1,svm,casb_ros,0.50,0.998537,0.979592,0.960000,1.000000,0.997076,0.998538,1.000000,1.000000,0.989064,NaN
9,42,1,knn,original,0.50,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,NaN
